# WEkEO Meteosat animation around Maroantsetra flood dates

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johrosa/srwi/blob/main/wekeo_meteosat_animation_maroantsetra_colab.ipynb)

This notebook uses the WEkEO Harmonized Data Access API (`hda`) to search and download Meteosat IODC products, then tries to render downloaded Meteosat files into frames and GIF animations.

WEkEO HDA requires a free WEkEO account. The query template below targets Meteosat High Rate SEVIRI IODC products and is intentionally easy to edit after copying the exact API request from the WEkEO Data Viewer.

## 1. Install and imports

In [ ]:
!pip -q install hda satpy imageio pillow pyproj pyresample netCDF4 h5netcdf

In [ ]:
import getpass
import json
import os
import re
import shutil
import zipfile
from pathlib import Path

import imageio.v2 as imageio
import pandas as pd
from hda import Client, Configuration
from IPython.display import Image, display
from PIL import Image as PILImage, ImageDraw, ImageFont

## 2. Parameters

In [ ]:
MAROANTSETRA_LON = 49.7333
MAROANTSETRA_LAT = -15.4333
BBOX = [47.4, -17.6, 52.1, -13.2]  # west, south, east, north

DATASET_ID = "EO:EUM:DAT:MSG:HRSEVIRI-IODC"

FLOOD_EVENTS = [
    {
        "name": "Cyclone Herold - Maroantsetra floods",
        "start": "2020-03-13T00:00:00.000Z",
        "end": "2020-03-18T23:59:59.999Z",
    },
    {
        "name": "Cyclone Gamane - north-east Madagascar floods",
        "start": "2024-03-26T00:00:00.000Z",
        "end": "2024-03-29T23:59:59.999Z",
    },
]

# Keep searches/downloads small first. Increase after confirming the query works.
MAX_PRODUCTS_PER_EVENT = 12
DOWNLOAD_DIR = Path("wekeo_meteosat_downloads")
FRAME_DIR = Path("wekeo_meteosat_frames")
GIF_DIR = Path("wekeo_meteosat_gifs")

# Satpy settings. For true day/night cloud motion, IR_108 is the practical first choice.
SATPY_READER = "seviri_l1b_native"
SATPY_DATASET = "IR_108"
GIF_FPS = 6

# If your WEkEO Data Viewer gives a different query, paste it in HDA_QUERY_OVERRIDES.
HDA_QUERY_OVERRIDES = {}

## 3. WEkEO credentials

In [ ]:
WEKEO_USERNAME = os.environ.get("WEKEO_USERNAME") or input("WEkEO username: ")
WEKEO_PASSWORD = os.environ.get("WEKEO_PASSWORD") or getpass.getpass("WEkEO password: ")

config = Configuration(user=WEKEO_USERNAME, password=WEKEO_PASSWORD)
hda_client = Client(config=config)
print("WEkEO HDA client ready")

## 4. Inspect dataset metadata

In [ ]:
dataset_info = hda_client.dataset(DATASET_ID)
display(dataset_info)

## 5. Search Meteosat products

In [ ]:
def build_query(event):
    query = {
        "dataset_id": DATASET_ID,
        "dtstart": event["start"],
        "dtend": event["end"],
        "bbox": BBOX,
        "itemsPerPage": MAX_PRODUCTS_PER_EVENT,
        "startIndex": 0,
    }
    query.update(HDA_QUERY_OVERRIDES)
    return query


def safe_event_name(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


search_rows = []
event_matches = {}

for event in FLOOD_EVENTS:
    query = build_query(event)
    print("Searching", event["name"])
    print(json.dumps(query, indent=2))
    matches = hda_client.search(query, MAX_PRODUCTS_PER_EVENT)
    event_matches[event["name"]] = matches
    print(matches)

    for item in getattr(matches, "results", []):
        search_rows.append({
            "event": event["name"],
            "id": item.get("id"),
            "start": item.get("startdate") or item.get("dtstart"),
            "end": item.get("enddate") or item.get("dtend"),
        })

search_df = pd.DataFrame(search_rows)
display(search_df.head(30))

## 6. Download products

In [ ]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

downloaded_dirs = []
for event in FLOOD_EVENTS:
    event_dir = DOWNLOAD_DIR / safe_event_name(event["name"])
    event_dir.mkdir(parents=True, exist_ok=True)
    matches = event_matches[event["name"]]
    print("Downloading", event["name"], "to", event_dir)

    # Slice first to avoid huge downloads while testing.
    matches[:MAX_PRODUCTS_PER_EVENT].download(download_dir=str(event_dir))
    downloaded_dirs.append(event_dir)

downloaded_files = []
for event_dir in downloaded_dirs:
    downloaded_files.extend([p for p in event_dir.rglob("*") if p.is_file()])

display(pd.DataFrame({"file": [str(p) for p in downloaded_files], "size_mb": [p.stat().st_size / 1e6 for p in downloaded_files]}).head(50))

## 7. Unpack archives

In [ ]:
def unpack_archives(root_dir):
    unpacked = []
    for zip_path in Path(root_dir).rglob("*.zip"):
        out_dir = zip_path.with_suffix("")
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(out_dir)
        unpacked.append(out_dir)
    return unpacked


unpacked_dirs = []
for event_dir in downloaded_dirs:
    unpacked_dirs.extend(unpack_archives(event_dir))

all_product_files = []
for event_dir in downloaded_dirs:
    all_product_files.extend([p for p in event_dir.rglob("*") if p.is_file()])

display(pd.DataFrame({"file": [str(p) for p in all_product_files]}).head(100))

## 8. Render frames with Satpy

In [ ]:
from satpy import Scene


def candidate_meteosat_files(event_dir):
    suffixes = {".nat", ".nat.gz", ".hrit", ".nc", ".bz2"}
    files = []
    for path in Path(event_dir).rglob("*"):
        if not path.is_file():
            continue
        lower = path.name.lower()
        if any(lower.endswith(s) for s in suffixes) or "seviri" in lower or "msg" in lower:
            files.append(path)
    return sorted(files)


def render_satpy_frame(product_files, out_png):
    scn = Scene(filenames=[str(p) for p in product_files], reader=SATPY_READER)
    scn.load([SATPY_DATASET])
    local = scn.crop(ll_bbox=(BBOX[0], BBOX[1], BBOX[2], BBOX[3]))
    local.save_dataset(SATPY_DATASET, filename=str(out_png))
    return out_png


def annotate_png(path, label):
    image = PILImage.open(path).convert("RGBA")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    margin = 10
    padding = 6
    bbox = draw.textbbox((0, 0), label, font=font)
    box = (margin, margin, margin + bbox[2] - bbox[0] + 2 * padding, margin + bbox[3] - bbox[1] + 2 * padding)
    draw.rectangle(box, fill=(0, 0, 0, 170))
    draw.text((margin + padding, margin + padding), label, fill=(255, 255, 255, 255), font=font)
    image.convert("RGB").save(path)


FRAME_DIR.mkdir(parents=True, exist_ok=True)
frame_rows = []

for event_dir in downloaded_dirs:
    event_name = event_dir.name
    files = candidate_meteosat_files(event_dir)
    print(event_name, "candidate files:", len(files))
    for idx, product_file in enumerate(files):
        out_png = FRAME_DIR / event_name / f"frame_{idx:04d}.png"
        out_png.parent.mkdir(parents=True, exist_ok=True)
        try:
            render_satpy_frame([product_file], out_png)
            annotate_png(out_png, f"{event_name} {product_file.name}")
            frame_rows.append({"event": event_name, "frame": str(out_png), "source": str(product_file)})
            print("rendered", out_png)
        except Exception as exc:
            print("Could not render", product_file, "->", exc)

frames_df = pd.DataFrame(frame_rows)
display(frames_df.head(50))

## 9. Build GIF animations

In [ ]:
GIF_DIR.mkdir(parents=True, exist_ok=True)
gif_paths = []

for event_name, group in frames_df.groupby("event") if not frames_df.empty else []:
    frame_paths = [Path(p) for p in group.sort_values("frame")["frame"]]
    if not frame_paths:
        continue
    images = [imageio.imread(path) for path in frame_paths]
    out_gif = GIF_DIR / f"{event_name}.gif"
    imageio.mimsave(out_gif, images, duration=1 / GIF_FPS)
    gif_paths.append(out_gif)
    print("GIF:", out_gif)
    display(Image(filename=str(out_gif)))

display(pd.DataFrame({"gif": [str(p) for p in gif_paths]}))

## 10. If rendering fails

Meteosat products can be distributed as native SEVIRI files, HRIT segments, NetCDF files, or compressed archives depending on the product and tailoring options. If Satpy cannot read the downloaded files:

1. Open the WEkEO Data Viewer for `EO:EUM:DAT:MSG:HRSEVIRI-IODC`.
2. Apply the same date and Maroantsetra bounding box.
3. Use "Show API request" and paste the JSON into `HDA_QUERY_OVERRIDES` or replace `build_query()`.
4. Prefer a tailored NetCDF or PNG-compatible product if available.

References: WEkEO HDA Python client docs: https://hda.readthedocs.io/en/latest/quickstart.html and WEkEO HDA help: https://help.wekeo.eu/en/articles/6751608-how-to-use-the-hda-api-in-python